<a href="https://colab.research.google.com/github/francesco-vaselli/NPTwins2026_Hackathon/blob/main/03_sampling_and_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 Notebook 3 — Evaluation, Improvement & Leaderboard

**Duration:** ~1 hour &nbsp;·&nbsp; **Prerequisites:** Notebook 2 (trained `model_nb2.pt`)

You have a working conditional Flow Matching model. The obvious next question is: **how good is it, really — and can we make it better?**

This notebook is structured as a **metric-driven improvement loop**:

1. 📏 Build a rigorous **scorecard** — Wasserstein distances, a classifier two-sample test, a b-tagging AUC delta — packaged as a single function.
2. 🧪 Score the **baseline** model from Notebook 2.
3. 🎛️ Try **inference-time** improvements (more ODE steps, Heun sampler). Re-score.
4. 🔁 Try a **retrain** with a **different probability path** (trigonometric / spherical). Re-score.
5. 🏆 Package your scorecard into a **leaderboard submission**.

### 🎯 Learning goals

- Know what "good" means for a generative detector surrogate — and how to quantify it.
- Feel the difference between **training-time** and **inference-time** choices.
- Touch the design space of Flow Matching (integrators, probability paths) concretely.
- Leave with a reproducible submission other students can beat. 🥇

### 🧪 How this notebook is structured

Same style as NB1/NB2:

- Each task is marked `✏️ Task X.Y` and followed by a `test_*` cell.
- The solution notebook `03_sampling_and_evaluation_solution.ipynb` lives next to this one — try first, peek only when stuck. 💪

### 📚 Symbols

| Symbol | Meaning |
|---|---|
| 🎯 | Goal of the section |
| ✏️ | Task to implement |
| 💡 | Hint |
| ⚠️ | Common pitfall |
| 🧪 | Validation cell |
| 🚀 | Let's run it! |
| 🎁 | Provided for you (just run) |
| 🏆 | Leaderboard step |


## ⚙️ Setup

> **Running on Google Colab?** Run the install cell — it grabs the deps, the NB3 test suite, `utils.py`, and (if needed) re-downloads `data.npy` and `model_nb2.pt`.
> **Running locally?** Make sure you've finished Notebook 2 so `model_nb2.pt` exists in this folder.

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch', 'numpy', 'scipy', 'scikit-learn', 'matplotlib'])
    import urllib.request
    for fname in ('test_evaluation.py', 'utils.py'):
        if not os.path.exists(fname):
            urllib.request.urlretrieve(
                f'https://raw.githubusercontent.com/fvaselli/NP_Twins/main/{fname}',
                fname,
            )
    print('✅ Colab environment ready.')
else:
    print('✅ Local environment — skipping install.')

if not os.path.exists('data.npy'):
    print('📥 Downloading CMS jet dataset...')
    import urllib.request
    urllib.request.urlretrieve(
        'https://zenodo.org/records/11126625/files/gen_ttbar_400k_final.npy',
        'data.npy',
    )
    print('✅ Dataset downloaded.')

assert os.path.exists('model_nb2.pt'), (
    'model_nb2.pt not found. Please run Notebook 2 end-to-end first '
    '(its last cell saves the checkpoint).'
)
print('✅ model_nb2.pt present.')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


class Preprocessor:
    # Preprocess the 3-variable reco target [btag, pt_ratio, N_const].
    # - btag:    standard scale
    # - pt:      compute ratio reco_pt / gen_pt, then standard scale
    # - N_const: dequantise with U[-0.5, 0.5], then standard scale
    # - gen:     standard-scale continuous features (pt, eta, phi, E);
    #            leave flavour + muonsInJet as-is (discrete condition flags).

    def __init__(self):
        self.reco_scaler = StandardScaler()
        self.gen_scaler = StandardScaler()

    def transform(self, reco, gen, fit=False):
        wreco, wgen = np.copy(reco), np.copy(gen)

        btag     = wreco[:, 0]
        pt_ratio = wreco[:, 1] / wgen[:, 0]
        n_const  = wreco[:, 4] + np.random.uniform(-0.5, 0.5, len(wreco))

        reco_3 = np.stack([btag, pt_ratio, n_const], axis=1)

        if fit:
            reco_3 = self.reco_scaler.fit_transform(reco_3)
            wgen[:, :4] = self.gen_scaler.fit_transform(wgen[:, :4])
        else:
            reco_3 = self.reco_scaler.transform(reco_3)
            wgen[:, :4] = self.gen_scaler.transform(wgen[:, :4])
        return reco_3, wgen

    def inverse_transform(self, reco_t, gen_t):
        wreco, wgen = np.copy(reco_t), np.copy(gen_t)
        wreco = self.reco_scaler.inverse_transform(wreco)
        wgen[:, :4] = self.gen_scaler.inverse_transform(wgen[:, :4])

        # pt_ratio → pt  (multiply by gen_pt)
        wreco[:, 1] = wreco[:, 1] * wgen[:, 0]
        # N_const: dequantised → integer count
        wreco[:, 2] = np.rint(wreco[:, 2])
        return wreco, wgen

Import the NB3 test suite (you'll run the `test_*` calls after each task).

In [ ]:
from test_evaluation import (
    test_compute_scorecard,
    test_heun_sample_reco,
    test_trig_fm_loss,
    run_all_tests_nb3,
)


## 0. Reload the Baseline Model 🧳

Nothing to implement — just run these cells to restore the Notebook 2 state: the model class, the trained weights, the preprocessor, and the validation set.

In [ ]:
# Model definition — identical to Notebook 2 (needed to load the state dict)

def sinusoidal_embedding(t, dim):
    half_dim = dim // 2
    freq = torch.exp(-torch.arange(half_dim, device=t.device, dtype=torch.float32)
                      * (np.log(10000.0) / half_dim))
    args = t * freq
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ConditionalVectorField(nn.Module):
    def __init__(self, reco_dim=3, cond_dim=6, time_dim=16, hidden_dim=128, n_blocks=3):
        super().__init__()
        self.time_dim = time_dim
        input_dim = reco_dim + time_dim + cond_dim
        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.SiLU())
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
                nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            )
            for _ in range(n_blocks)
        ])
        self.output_proj = nn.Linear(hidden_dim, reco_dim)

    def forward(self, x_t, t, cond):
        h = torch.cat([x_t, sinusoidal_embedding(t, self.time_dim), cond], dim=-1)
        h = self.input_proj(h)
        for block in self.blocks:
            h = h + block(h)
        return self.output_proj(h)


In [ ]:
# Load the NB2 checkpoint

checkpoint = torch.load('model_nb2.pt', map_location=device, weights_only=False)
cfg = checkpoint['model_config']
model = ConditionalVectorField(**cfg).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
preprocessor = checkpoint['preprocessor']
reco_dim = cfg['reco_dim']
cond_dim = cfg['cond_dim']

print(f'Model loaded: reco_dim={reco_dim}, cond_dim={cond_dim}, '
      f'{sum(p.numel() for p in model.parameters()):,} params')


In [ ]:
# Rebuild the same 80/20 train/val split as NB2 (same seed → identical partition)

class DataExtractor:
    def __init__(self, data_path, n_samples=None):
        self.data = np.load(data_path, allow_pickle=True)
        if n_samples is not None:
            self.data = self.data[:n_samples]
        self.reco_features = ['btag', 'pt', 'phi', 'eta', 'N_const', 'ctag']
        self.reco = self.data[:, [5, 6, 7, 8, 10, 19]]
        self.gen_features = ['pt', 'eta', 'phi', 'E', 'flavour', 'muonsInJet']
        self.gen = self.data[:, [0, 1, 2, 3, 4, 9]]
        flav = np.abs(self.gen[:, 4])
        flav = np.where(np.isin(flav, [1, 2, 3, 21]), 0, flav)
        flav = np.where(flav == 4, 1, flav)
        flav = np.where(flav == 5, 2, flav)
        self.gen[:, 4] = flav
    def get_reco(self): return self.reco
    def get_gen(self):  return self.gen


extractor = DataExtractor('data.npy', n_samples=400000)
reco_raw = extractor.get_reco()
gen_raw  = extractor.get_gen()

reco_tr_raw, reco_vl_raw, gen_tr_raw, gen_vl_raw = train_test_split(
    reco_raw, gen_raw, test_size=0.2, random_state=42
)
# Use the fitted preprocessor from NB2 — do NOT re-fit.
reco_train, gen_train = preprocessor.transform(reco_tr_raw, gen_tr_raw, fit=False)
reco_val,   gen_val   = preprocessor.transform(reco_vl_raw, gen_vl_raw, fit=False)

reco_train_t = torch.tensor(reco_train, dtype=torch.float32).to(device)
gen_train_t  = torch.tensor(gen_train,  dtype=torch.float32).to(device)
reco_val_t   = torch.tensor(reco_val,   dtype=torch.float32).to(device)
gen_val_t    = torch.tensor(gen_val,    dtype=torch.float32).to(device)

print(f'Train: reco {reco_train_t.shape}, gen {gen_train_t.shape}')
print(f'Val:   reco {reco_val_t.shape}, gen {gen_val_t.shape}')


In [ ]:
# Euler sampler from NB2 — we re-include it here so this notebook is self-contained.

@torch.no_grad()
def generate_reco(model, gen_cond, n_steps=100, batch_size=4096):
    model.eval()
    reco_dim = model.output_proj.out_features
    h = 1.0 / n_steps
    all_samples = []
    for start in range(0, gen_cond.shape[0], batch_size):
        cond_chunk = gen_cond[start:start + batch_size]
        n = cond_chunk.shape[0]
        x = torch.randn(n, reco_dim, device=gen_cond.device)
        for step in range(n_steps):
            t = torch.full((n, 1), step * h, device=gen_cond.device)
            x = x + h * model(x, t, cond_chunk)
        all_samples.append(x)
    return torch.cat(all_samples, dim=0)


# --- Helper: run the model end-to-end and return physical-space arrays -------
def sample_and_inverse(model, sampler, gen_val_t, gen_val_pp, reco_val_pp,
                        n_steps=100, batch_size=4096):
    # Returns (generated_phys, target_phys, gen_phys) — all numpy, ready for scoring.
    gen_scaled = sampler(model, gen_val_t, n_steps=n_steps, batch_size=batch_size).cpu().numpy()
    generated_phys, gen_phys = preprocessor.inverse_transform(gen_scaled, gen_val_pp)
    target_phys,    _        = preprocessor.inverse_transform(reco_val_pp, gen_val_pp)
    return generated_phys, target_phys, gen_phys


## 1. The Scorecard 📏

Throughout the notebook we'll keep grading our model on a fixed set of metrics. To make re-scoring cheap and consistent, we wrap them in a single function.

### 🎛️ Three metrics, three angles

| Metric | Reads like… | Good when it is… |
|---|---|---|
| **Wasserstein distance per feature** `ws_per_feature` + sum | "Average ground distance" between the generated and target 1-D histograms. Units: same as the feature. | Small (≈ 0). |
| **Classifier Two-Sample Test** `c2st` = \|AUC − 0.5\| | Trains a BDT to tell model samples from real samples. 0 ⇒ indistinguishable. | Small (≈ 0). |
| **b-tag AUC delta** `auc_delta_btag` | How much does the b-vs-light AUC computed from **generated** btag scores differ from the AUC on **target** btag scores? | Small (≈ 0). |

The **`ws_sum`** will be our single leaderboard number.

### ✏️ Task 1.1 — Implement `compute_scorecard`

**Specification**
```python
compute_scorecard(generated, target, gen,
                  feature_names=None, c2st_n=20000, random_state=42)
    -> dict with keys:
        ws_per_feature  : list[float]  (length = reco_dim)
        ws_sum          : float
        c2st            : float  ( = |BDT AUC − 0.5| )
        auc_delta_btag  : float  ( = |AUC_target − AUC_model| )
```

**💡 Hints**
- `scipy.stats.wasserstein_distance(a, b)` computes 1-D WS between two 1-D arrays.
- For C2ST, concatenate `target[:n]` (label 1) and `generated[:n]` (label 0), then fit a `HistGradientBoostingClassifier(max_iter=100, random_state=random_state)` on a 70/30 split. Use `roc_auc_score` on the held-out 30%. `c2st = |AUC − 0.5|`.
- For the b-tag AUC delta, mask to jets with **flavour ∈ {0, 2}** (light + bottom), label b-jets as 1, and use the **btag score (column 0)** as the classifier output. Compute the ROC AUC twice — once with `target[:, 0]` as the score, once with `generated[:, 0]`. Return `|AUC_target − AUC_model|`.
- `c2st_n` caps the number of rows fed to the BDT — keeps repeated scoring fast.

**⚠️ Pitfalls**
- If your generator crashes on `flavour` values that include `1` (charm) — remember to apply the mask `(flav == 2) | (flav == 0)` first.
- Don't pass the full 80 k validation set to the BDT; it'll be slow. `c2st_n = 20000` is plenty.


In [ ]:
from scipy.stats import wasserstein_distance
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split


def compute_scorecard(generated, target, gen,
                      feature_names=None, c2st_n=20000, random_state=42):
    # Build a dict of evaluation metrics for a (generated, target, gen) triple.
    #
    # Args
    # ----
    # generated : np.ndarray (N, reco_dim) — model samples in physical space
    # target    : np.ndarray (N, reco_dim) — real reco samples in physical space
    # gen       : np.ndarray (N, cond_dim) — gen-level conditions (flavour at col 4)
    # c2st_n    : int — max rows for the BDT (for speed)
    #
    # Returns
    # -------
    # dict with keys: ws_per_feature, ws_sum, c2st, auc_delta_btag

    # YOUR CODE HERE
    # 1. Compute per-feature Wasserstein distances and their sum.
    # 2. Run the BDT-based C2ST on the first c2st_n rows of each.
    # 3. Compute the b-tag AUC delta on jets with flavour in {0, 2}
    #    using column 0 as the score.
    # 4. Return {ws_per_feature, ws_sum, c2st, auc_delta_btag}.
    raise NotImplementedError("Build the scorecard! (Task 1.1)")


In [ ]:
test_compute_scorecard(compute_scorecard);


## 2. Baseline Scorecard 🧪

Let's measure the NB2 model — this is the number you're trying to beat.

In [ ]:
reco_3_names = ['btag', r'$p_T$ [GeV]', r'$N_{\mathrm{const}}$']

print('Sampling (Euler, 100 steps) ...')
gen_phys, tgt_phys, gen_val_phys = sample_and_inverse(
    model, generate_reco, gen_val_t, gen_val, reco_val, n_steps=100,
)
print('Scoring ...')
baseline_scorecard = compute_scorecard(gen_phys, tgt_phys, gen_val_phys)


def print_scorecard(name, sc, compare_to=None):
    print('─' * 60)
    print(f'Scorecard — {name}')
    print('─' * 60)
    for i, v in enumerate(sc['ws_per_feature']):
        label = reco_3_names[i] if i < len(reco_3_names) else f'feat {i}'
        extra = ''
        if compare_to is not None:
            delta = v - compare_to['ws_per_feature'][i]
            extra = f'   Δ={delta:+.4f}'
        print(f'  WS [{label:<18}]: {v:.4f}{extra}')
    extras = {}
    for key in ('ws_sum', 'c2st', 'auc_delta_btag'):
        v = sc[key]
        delta = v - compare_to[key] if compare_to else None
        extras[key] = (v, delta)
        if delta is not None:
            print(f'  {key:<22}: {v:.4f}   Δ={delta:+.4f}')
        else:
            print(f'  {key:<22}: {v:.4f}')
    print()
    return extras


print_scorecard('baseline (NB2, Euler 100 steps)', baseline_scorecard);


Keep those three numbers in mind — every improvement below will update the delta column. A negative `Δ` means the metric improved. ⬇️🟢

## 🎯 Your Mission: Beat the Baseline

The number above (`ws_sum` for the NB2 model) is what you're trying to beat. Sections **3** and **4** below walk you through a handful of improvements **already implemented for you** — an ODE-step sweep, the conditional Heun sampler, and a retrain with a trigonometric probability path. Run them, watch the deltas move, get a feel for the metric-driven loop.

But here's the catch: **everyone runs these.** A "trig path + Heun 100" submission is the floor of the leaderboard, not the top. The provided ideas are warm-ups — they'll show you which knobs matter and how to read the scorecard, but they won't be enough on their own to put you ahead.

The edge over other participants comes from what you do **after** §4. The §7 wrap-up has a starter list (bigger model, longer training, classifier guidance, …) but anything is fair game as long as the scorecard is computed honestly on the val split. Bring your own idea, retrain, re-score, and submit your best run.

## 3. Inference-Time Improvements 🎛️

These are the **free** knobs — same trained weights, different way of using them.

### 3.1 — Sweep the number of Euler steps 🔢

The ODE's global error is $\mathcal{O}(1/N)$ for Euler — more steps should help, at linear extra cost. Let's see.

In [ ]:
step_counts = [5, 10, 25, 50, 100, 200]
ws_sum_vs_steps = []

for n_steps in step_counts:
    gp, tp, gvp = sample_and_inverse(
        model, generate_reco, gen_val_t, gen_val, reco_val, n_steps=n_steps,
    )
    sc = compute_scorecard(gp, tp, gvp, c2st_n=5000)  # smaller BDT for the sweep
    ws_sum_vs_steps.append(sc['ws_sum'])
    print(f'  n_steps={n_steps:4d} | ws_sum = {sc["ws_sum"]:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(step_counts, ws_sum_vs_steps, 'o-', color='steelblue', lw=2)
ax.set_xlabel('Number of Euler steps'); ax.set_ylabel('ws_sum')
ax.set_xscale('log'); ax.set_title('Sample quality vs ODE steps (Euler)')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


💭 **Observation.** `ws_sum` typically drops sharply from 5 → 25 steps and then plateaus — beyond ~50 steps, the residual error is dominated by the *model* (not the integrator). More steps won't help if the vector field itself is wrong.

### 3.2 — Heun's method with conditioning 🏃‍♀️

Heun's method (two-stage Runge–Kutta) — you implemented it for the unconditional case in Notebook 1. The conditional version is exactly the same, except the condition `c` rides along at every evaluation:

$$k_1 = u_\theta(x, t, c),\qquad k_2 = u_\theta(x + h k_1,\, t+h,\, c),\qquad x \leftarrow x + \tfrac{h}{2}(k_1 + k_2)$$

Because each Heun step costs **two** model evaluations, the fair comparison at fixed compute is Heun with `N/2` steps vs Euler with `N` steps.

### ✏️ Task 3.2 — Implement `heun_sample_reco`

**Specification** — same interface as `generate_reco`:
```python
@torch.no_grad()
heun_sample_reco(model, gen_cond, n_steps=100, batch_size=4096) -> (N, reco_dim) tensor
```
Integrate from `t=0` to `t=1` with step `h = 1/n_steps`, processing `gen_cond` in chunks of up to `batch_size` rows.

**💡 Hints**
- Start from `torch.randn(n, reco_dim, device=gen_cond.device)`.
- At each step: compute `k1`, the "predicted next x", then `k2` at `(x + h*k1, t+h)`, then average.
- `reco_dim = model.output_proj.out_features` saves passing it in.


In [ ]:
@torch.no_grad()
def heun_sample_reco(model, gen_cond, n_steps=100, batch_size=4096):
    # Heun (RK2) sampler — conditional version.
    # Returns tensor of shape (N, reco_dim).

    # YOUR CODE HERE
    # - reco_dim = model.output_proj.out_features
    # - loop over chunks of size batch_size
    # - per step:
    #     k1 = model(x, t,     cond)
    #     k2 = model(x + h*k1, t+h,  cond)
    #     x  = x + 0.5 * h * (k1 + k2)
    raise NotImplementedError("Implement the conditional Heun sampler! (Task 3.2)")


In [ ]:
test_heun_sample_reco(heun_sample_reco, model_cls=ConditionalVectorField);


#### Same compute, Euler vs Heun

We compare at **equal NFE** (network forward evaluations): Euler with `N` steps ≡ Heun with `N/2` steps.

In [ ]:
nfe_budgets = [10, 20, 40, 80]
rows = []
for nfe in nfe_budgets:
    gp_e, tp, gvp = sample_and_inverse(
        model, generate_reco,      gen_val_t, gen_val, reco_val, n_steps=nfe,
    )
    ws_e = compute_scorecard(gp_e, tp, gvp, c2st_n=5000)['ws_sum']
    gp_h, _,  _   = sample_and_inverse(
        model, heun_sample_reco,   gen_val_t, gen_val, reco_val, n_steps=max(1, nfe // 2),
    )
    ws_h = compute_scorecard(gp_h, tp, gvp, c2st_n=5000)['ws_sum']
    rows.append((nfe, ws_e, ws_h))
    print(f'NFE={nfe:3d} | Euler ws_sum={ws_e:.4f} | Heun ws_sum={ws_h:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in rows], [r[1] for r in rows], 'o-', label='Euler', color='steelblue')
ax.plot([r[0] for r in rows], [r[2] for r in rows], 's-', label='Heun',  color='tomato')
ax.set_xlabel('NFE (model forward passes)'); ax.set_ylabel('ws_sum')
ax.set_title('Euler vs Heun at equal compute budget')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


At very low NFE Heun usually wins; at high NFE the gap shrinks because both are dominated by model error. ⚖️

#### Pick your inference-time champion 🏆

Let's lock in the configuration you'll carry forward: Heun with 100 inner steps (200 NFE).

In [ ]:
gp_heun, tp_heun, gvp_heun = sample_and_inverse(
    model, heun_sample_reco, gen_val_t, gen_val, reco_val, n_steps=100,
)
heun_scorecard = compute_scorecard(gp_heun, tp_heun, gvp_heun)
print_scorecard('Heun (100 steps, NFE=200)', heun_scorecard, compare_to=baseline_scorecard);


## 4. 🔁 Retrain with a Trigonometric Probability Path

Inference-time tricks only take you so far. The **training-time** design choice with the biggest single-knob impact is the **probability path** — how $x_t$ interpolates between noise and data.

NB1 and NB2 used the **linear (conditional OT)** path:

$$x_t^{\text{lin}} = (1 - t) x_0 + t x_1,\qquad v_t^{\text{lin}} = x_1 - x_0$$

A classic alternative is the **trigonometric / spherical** path, which follows a circular arc in sample space:

$$x_t^{\text{trig}} = \cos\!\big(\tfrac{\pi t}{2}\big)\, x_0 + \sin\!\big(\tfrac{\pi t}{2}\big)\, x_1,\qquad
   v_t^{\text{trig}} = \tfrac{\pi}{2}\big(\cos\!\big(\tfrac{\pi t}{2}\big)\, x_1 - \sin\!\big(\tfrac{\pi t}{2}\big)\, x_0\big)$$

Why try this?

- **Variance-preserving**: if $x_0, x_1 \sim \mathcal{N}(0, I)$ independently, then $\text{Var}(x_t^{\text{trig}}) = \cos^2(\pi t/2) + \sin^2(\pi t/2) = 1$ at every $t$ — the *scale* of the intermediate state is constant. Compare with the linear path, where $\text{Var}(x_t^{\text{lin}}) = (1-t)^2 + t^2$ dips to $\tfrac{1}{2}$ at $t = \tfrac{1}{2}$.
- **Same model, different teacher**: we're **not** changing the network architecture — just what we ask it to regress. So we get to reuse everything else.

### ✏️ Task 4.1 — Implement `trig_fm_loss`

**Specification** — identical signature to `conditional_fm_loss`:
```python
trig_fm_loss(model, reco_batch, gen_batch) -> scalar tensor
```

**💡 Hints**
- `phi = (np.pi / 2) * t` — compute once per batch.
- `x_t = cos(phi) * x0 + sin(phi) * reco_batch`
- `target = (np.pi / 2) * (cos(phi) * reco_batch - sin(phi) * x0)`
- Return `mean((model(x_t, t, gen) - target) ** 2)`

**⚠️ Pitfall** — `t` must have shape `(batch, 1)` so that `cos(phi)` broadcasts correctly with `(batch, reco_dim)` tensors.


In [ ]:
def trig_fm_loss(model, reco_batch, gen_batch):
    # Conditional Flow Matching loss under the trigonometric probability path.
    #
    # Args:
    #   model:      ConditionalVectorField
    #   reco_batch: shape (B, reco_dim)
    #   gen_batch:  shape (B, cond_dim)
    # Returns:
    #   scalar MSE loss.

    # YOUR CODE HERE
    # 1. x0 = torch.randn_like(reco_batch)
    # 2. t  ~ Uniform[0, 1]  shape (batch, 1) on reco_batch.device
    # 3. phi = (pi/2) * t
    # 4. x_t    = cos(phi) * x0 + sin(phi) * reco_batch
    # 5. target = (pi/2) * (cos(phi) * reco_batch - sin(phi) * x0)
    # 6. return mean((model(x_t, t, gen_batch) - target) ** 2)
    raise NotImplementedError("Implement the trigonometric-path loss! (Task 4.1)")


In [ ]:
test_trig_fm_loss(trig_fm_loss, model_cls=ConditionalVectorField);


### 🚀 Retrain with the new path

We reuse the NB2 architecture and training loop — only the loss function changes. 60 epochs is enough to see a clear effect (about 3 minutes on CPU, seconds on GPU). ☕

In [ ]:
def train_with_loss(model, loss_fn, reco_train, gen_train, reco_val, gen_val,
                    n_epochs=60, batch_size=512, lr=1e-3, log_every=10):
    # Generic training loop that takes a loss function — so we can swap linear vs trig.
    optimizer = optim.Adam(model.parameters(), lr=lr)
    n_train = reco_train.shape[0]
    train_losses, val_losses = [], []

    for epoch in range(n_epochs):
        model.train()
        idx = torch.randperm(n_train, device=reco_train.device)
        ep_loss, n_b = 0.0, 0
        for i in range(0, n_train, batch_size):
            b = idx[i:i + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model, reco_train[b], gen_train[b])
            loss.backward()
            optimizer.step()
            ep_loss += loss.item(); n_b += 1
        train_losses.append(ep_loss / n_b)

        model.eval()
        with torch.no_grad():
            vs = min(5000, reco_val.shape[0])
            val_losses.append(loss_fn(model, reco_val[:vs], gen_val[:vs]).item())

        if (epoch + 1) % log_every == 0:
            print(f'  epoch {epoch+1:3d}/{n_epochs}  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}')

    return train_losses, val_losses


print('Training a fresh model with the trigonometric path ...')
torch.manual_seed(0)
model_trig = ConditionalVectorField(**cfg).to(device)
_ = train_with_loss(
    model_trig, trig_fm_loss,
    reco_train_t, gen_train_t, reco_val_t, gen_val_t,
    n_epochs=60, batch_size=512, lr=1e-3,
)
print('✅ Training complete.')


Score the trigonometric model with the **best inference setup** we found (Heun, 100 steps).

In [ ]:
gp_trig, tp_trig, gvp_trig = sample_and_inverse(
    model_trig, heun_sample_reco, gen_val_t, gen_val, reco_val, n_steps=100,
)
trig_scorecard = compute_scorecard(gp_trig, tp_trig, gvp_trig)
print_scorecard('Trigonometric path + Heun 100', trig_scorecard, compare_to=baseline_scorecard);


#### Diagnostic: 1-D histograms side-by-side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, (ax, name) in enumerate(zip(axes, reco_3_names)):
    kw = dict(bins=60, histtype='step', lw=2, density=True)
    ax.hist(tp_heun[:, i],  color='black',     label='Target',           **kw)
    ax.hist(gp_heun[:, i],  color='steelblue', label='Baseline+Heun',    **kw)
    ax.hist(gp_trig[:, i],  color='tomato',    label='Trig+Heun',        **kw)
    ax.set_title(name); ax.legend(fontsize=9)
    if i > 0:
        ax.set_yscale('log')
plt.suptitle('Baseline vs Trig path — Heun sampling at 100 steps', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


💭 Typical outcome: the trig-path model matches or slightly beats the linear-path baseline on `ws_sum`, often with visibly crisper tails on `pt` and `btag`. Results will vary with seed and training length — that's exactly what the leaderboard captures.

## 5. 🏆 Your Leaderboard Submission

Time to serialize your best scorecard into a file you can submit. We save a JSON with:

- a **submission id** (your name + a short run tag)
- the **config** that produced the numbers (path, sampler, n_steps)
- the **full scorecard** (each metric)

> **Rules of the game.** The public leaderboard is keyed on **`ws_sum`** (lower = better). You submit by opening a PR that adds **two** files keyed on the same `<id>`:
> - `submissions/scorecards/<id>.json` — the JSON written by the cell below.
> - `submissions/notebooks/<id>.ipynb` — a renamed copy of *your* run of this notebook with cells executed (outputs visible), so a reviewer can audit the numbers you're claiming.
>
> Crucially, your submission **does not overwrite** the upstream `03_sampling_and_evaluation.ipynb` — that one stays as the canonical baseline for the next cohort. More on the PR workflow at the end.

### ✏️ Task 5.1 — Fill in your submission metadata and save

Edit the `submission` dict below with **your name**, a short **run tag**, and the configuration that produced your best run. The default here submits the trigonometric-path + Heun run; feel free to point it at a different one (e.g. a longer-trained model, bigger hidden size, etc.).

In [ ]:
import json, time

# ⬇️ EDIT ME
your_name = 'your-github-handle'
run_tag   = 'trig-heun-100'
notes     = 'baseline NB2 arch retrained with trig path; Heun @ 100 steps'

# Pick the scorecard and config you want to submit:
final_scorecard = trig_scorecard         # or: baseline_scorecard / heun_scorecard
final_config    = {
    'path':       'trigonometric',       # 'linear' or 'trigonometric'
    'sampler':    'heun',                # 'euler' or 'heun'
    'n_steps':    100,
    'arch':       cfg,                   # model_config dict from NB2
    'n_epochs':   60,
}

submission = {
    'id':        f'{your_name}__{run_tag}',
    'name':      your_name,
    'run_tag':   run_tag,
    'notes':     notes,
    'timestamp': int(time.time()),
    'config':    final_config,
    'scorecard': final_scorecard,
}

os.makedirs('submissions/scorecards', exist_ok=True)
os.makedirs('submissions/notebooks',  exist_ok=True)
out_path = f'submissions/scorecards/{submission["id"]}.json'
with open(out_path, 'w') as f:
    json.dump(submission, f, indent=2)

print(f'✅ Wrote {out_path}')
print(f'   ws_sum         : {final_scorecard["ws_sum"]:.4f}   (lower = better)')
print(f'   c2st           : {final_scorecard["c2st"]:.4f}')
print(f'   auc_delta_btag : {final_scorecard["auc_delta_btag"]:.4f}')
print()
print(f'📓 Next: save THIS notebook (with cells executed) as')
print(f'        submissions/notebooks/{submission["id"]}.ipynb')
print(f'   then open a PR that adds both files. Do NOT overwrite the upstream')
print(f'   03_sampling_and_evaluation.ipynb — leave that one untouched.')


## 6. Side-by-Side Comparison

All three runs on one grid — pick the row you like best (it becomes your submission).

In [ ]:
rows = [
    ('Baseline (Euler 100)',     baseline_scorecard),
    ('Heun 100 (same weights)',  heun_scorecard),
    ('Trig path + Heun 100',     trig_scorecard),
]

print('─' * 78)
print(f'{"Run":<32} {"ws_sum":>10} {"c2st":>10} {"AUC Δ btag":>14}')
print('─' * 78)
for name, sc in rows:
    print(f'{name:<32} {sc["ws_sum"]:>10.4f} {sc["c2st"]:>10.4f} {sc["auc_delta_btag"]:>14.4f}')
print('─' * 78)


## 7. Final Test Sweep ✅

In [ ]:
run_all_tests_nb3(
    compute_scorecard=compute_scorecard,
    heun_sample_reco=heun_sample_reco,
    trig_fm_loss=trig_fm_loss,
    model_cls=ConditionalVectorField,
);


## 🏁 Summary & Leaderboard How-To

In this notebook you:

1. Wrapped three complementary metrics (Wasserstein, C2ST, b-tag AUC delta) into a **reusable scorecard** and scored the NB2 baseline.
2. Swept **ODE step counts** and implemented a **conditional Heun sampler** — felt the inference-time vs compute trade-off.
3. Swapped the **linear** probability path for the **trigonometric** path, retrained, and compared.
4. Saved a JSON **submission** containing your best scorecard.

### 🏆 How the public leaderboard works

The `submissions/` folder in the repo is the single source of truth, with two subfolders:

```
submissions/
  scorecards/<id>.json     ← the numbers (what gets ranked)
  notebooks/<id>.ipynb     ← the executed notebook (what gets audited)
```

To enter the leaderboard:

1. **Fork** the repo on GitHub.
2. **Run this notebook end-to-end** so all cells have outputs (the `print_scorecard(...)` cells are your evidence).
3. **Save a copy of this notebook** as `submissions/notebooks/<your-id>.ipynb` — *with cells executed*. Don't clear outputs (`File → Save and Checkpoint` after running keeps them). **Do not overwrite the upstream `03_sampling_and_evaluation.ipynb`** — that file is the canonical baseline for the next cohort and stays untouched.
4. **Commit both files** on a new branch: `submissions/scorecards/<your-id>.json` and `submissions/notebooks/<your-id>.ipynb`.
5. **Open a Pull Request** back to `main`. A maintainer eyeballs the notebook output against the JSON; if the numbers match and the code isn't obviously cheating (e.g. `generated = reco_val` 🙃), the PR is merged.
6. On merge, a small GitHub Action collects all `submissions/scorecards/*.json`, sorts by `ws_sum`, and rewrites `LEADERBOARD.md`. Your notebook stays in `submissions/notebooks/` as a permanent record.

> **Why trust-based?** Because the scorecard runs locally on your validation split, a PR-side re-score would only verify what you put in the JSON (the CI has nothing fresh to score against unless it re-trains your model, which is expensive). We lean on the executed notebook as the audit trail instead — it's a hackathon, not ImageNet.
>
> **What counts as cheating?** Running the scorecard on `target` instead of `generated`, or against your training data. Both are trivially spotted from the notebook. Tuning hyperparameters against `ws_sum` on the val split is *fine* — it's what you're supposed to do.

### 🚀 Ideas for a better submission

All of these are legal entries:

- **More epochs / bigger model** — `hidden_dim=256`, `n_blocks=5`, `n_epochs=150`.
- **Different path** — try variance-exploding (linear but with noise schedule) or cosine.
- **Lower learning rate + longer run** — can clean up tails.
- **Warm-up** from the NB2 checkpoint instead of training from scratch.
- **Classifier-guided sampling** — train a BDT classifier and use its gradient at inference.

Have fun, and may the lowest `ws_sum` win. 🥇
